<a href="https://colab.research.google.com/github/sdsd38931/raz/blob/main/1lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from collections import defaultdict
import gdown

def load_sessions(file_path):
    sessions = []
    with open(file_path, 'r') as file:
        for line in file:
            viewed, bought = line.strip().split(';')
            viewed_items = list(map(int, viewed.split(',')))
            bought_items = list(map(int, bought.split(','))) if bought else []
            sessions.append((viewed_items, bought_items))
    return sessions

def calculate_frequencies(sessions):
    view_counts = defaultdict(int)
    purchase_counts = defaultdict(int)

    for viewed, bought in sessions:
        for item in viewed:
            view_counts[item] += 1
        for item in bought:
            purchase_counts[item] += 1

    return view_counts, purchase_counts

def recommend_by_views(viewed_items, view_counts):
    sorted_items = sorted(viewed_items, key=lambda x: (-view_counts[x], viewed_items.index(x)))
    return sorted_items

def recommend_by_purchases(viewed_items, purchase_counts):
    sorted_items = sorted(viewed_items, key=lambda x: (-purchase_counts[x], viewed_items.index(x)))
    return sorted_items

def evaluate_recommendations(recommendations, actual_buys):
    hits = sum(1 for item in recommendations if item in actual_buys)
    precision = hits / len(recommendations) if recommendations else 0
    recall = hits / len(actual_buys) if actual_buys else 0
    return precision, recall

# Ссылки на файлы на Google Диске (замените на ваши ссылки)
train_file_url = 'https://drive.google.com/uc?id=1Ev9yqQYyfVefe0ZA2cVfxsRZLAo9PzW0'
test_file_url = 'https://drive.google.com/uc?id=1O0le5x6BPC0MuEl4Xv2cY9PCJksqHccM'

# Загрузка данных
gdown.download(train_file_url, 'sessions_train.txt', quiet=False)
gdown.download(test_file_url, 'sessions_test.txt', quiet=False)

train_sessions = load_sessions('sessions_train.txt')
test_sessions = load_sessions('sessions_test.txt')

# Расчет частот
view_counts_train, purchase_counts_train = calculate_frequencies(train_sessions)

# Рекомендации и оценка для обучающей выборки
train_results = []
for viewed, bought in train_sessions:
    if not bought:
        continue
    recommendations_views = recommend_by_views(viewed, view_counts_train)[:5]
    recommendations_purchases = recommend_by_purchases(viewed, purchase_counts_train)[:5]

    precision_views, recall_views = evaluate_recommendations(recommendations_views[:1], bought)
    precision_purchases, recall_purchases = evaluate_recommendations(recommendations_purchases[:1], bought)

    train_results.append((precision_views, recall_views, precision_purchases, recall_purchases))

# Результаты по обучающей выборке
avg_precision_views_train = sum(result[0] for result in train_results) / len(train_results)
avg_recall_views_train = sum(result[1] for result in train_results) / len(train_results)
avg_precision_purchases_train = sum(result[2] for result in train_results) / len(train_results)
avg_recall_purchases_train = sum(result[3] for result in train_results) / len(train_results)

# Рекомендации и оценка для тестовой выборки
test_results = []
for viewed, bought in test_sessions:
    if not bought:
        continue
    recommendations_views = recommend_by_views(viewed, view_counts_train)[:5]
    recommendations_purchases = recommend_by_purchases(viewed, purchase_counts_train)[:5]

    precision_views, recall_views = evaluate_recommendations(recommendations_views[:1], bought)
    precision_purchases, recall_purchases = evaluate_recommendations(recommendations_purchases[:1], bought)

    test_results.append((precision_views, recall_views, precision_purchases, recall_purchases))

# Результаты по тестовой выборке
avg_precision_views_test = sum(result[0] for result in test_results) / len(test_results)
avg_recall_views_test = sum(result[1] for result in test_results) / len(test_results)
avg_precision_purchases_test = sum(result[2] for result in test_results) / len(test_results)
avg_recall_purchases_test = sum(result[3] for result in test_results) / len(test_results)

# Вывод результатов
print(f"Train - Precision Views: {avg_precision_views_train:.2f}, Recall Views: {avg_recall_views_train:.2f}")
print(f"Train - Precision Purchases: {avg_precision_purchases_train:.2f}, Recall Purchases: {avg_recall_purchases_train:.2f}")
print(f"Test - Precision Views: {avg_precision_views_test:.2f}, Recall Views: {avg_recall_views_test:.2f}")
print(f"Test - Precision Purchases: {avg_precision_purchases_test:.2f}, Recall Purchases: {avg_recall_purchases_test:.2f}")



Downloading...
From: https://drive.google.com/uc?id=1Ev9yqQYyfVefe0ZA2cVfxsRZLAo9PzW0
To: /content/sessions_train.txt
100%|██████████| 2.06M/2.06M [00:00<00:00, 150MB/s]
Downloading...
From: https://drive.google.com/uc?id=1O0le5x6BPC0MuEl4Xv2cY9PCJksqHccM
To: /content/sessions_test.txt
100%|██████████| 2.06M/2.06M [00:00<00:00, 158MB/s]


Train - Precision Views: 0.51, Recall Views: 0.44
Train - Precision Purchases: 0.80, Recall Purchases: 0.69
Test - Precision Views: 0.48, Recall Views: 0.42
Test - Precision Purchases: 0.53, Recall Purchases: 0.46
